# Assignment 22: Embeddings Models, Vector store and Similarity Search

# Part 1: Embeddings Models

## TASK 1: OpenAI Emebeddings Model

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

In [4]:
load_dotenv()

True

In [5]:
document_chunks = [
    "Python is a high-level programming language.",
    "Machine learning allows computers to learn patterns from data.",
    "RAG combines retrieval with large language models.",
    "Vector databases store embeddings for similarity search.",
    "LangChain is a framework for building LLM applications."
]

In [7]:
len(document_chunks)

5

In [8]:
openai_embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)


In [9]:
openai_vectors = openai_embeddings.embed_documents(
    document_chunks
)

In [10]:
len(openai_vectors)

5

In [11]:
openai_vectors[0][:10]

[-0.007289886474609375,
 0.009063720703125,
 0.0166168212890625,
 0.003414154052734375,
 0.04425048828125,
 -0.035858154296875,
 -0.0276641845703125,
 0.038360595703125,
 -0.0004124641418457031,
 0.00537109375]

## TASK 2: Hugging Face Embeddings Models

In [13]:
from langchain_huggingface import HuggingFaceEmbeddings


In [16]:
hf_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

c:\Users\arunk\anaconda3\envs\genai_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3316.90it/s]


In [18]:

hf_vectors = hf_embeddings.embed_documents(
    document_chunks
)

In [19]:

print("Number of embeddings:", len(hf_vectors))


Number of embeddings: 5


In [20]:
print("Embedding vector length:",
      len(hf_vectors[0]))

Embedding vector length: 384


In [21]:
print(hf_vectors[0][:10])

[-0.044812463223934174, -0.011056646704673767, -0.02356061153113842, -0.01303253136575222, -0.04396739974617958, -0.1665979027748108, -0.03483545780181885, 0.08270729333162308, -0.07906714081764221, -0.015830401331186295]


In [23]:
import time

In [24]:
start_time = time.perf_counter()

openai_vectors = openai_embeddings.embed_documents(
    document_chunks
)

openai_time = time.perf_counter() - start_time

In [25]:

start_time = time.perf_counter()

hf_vectors = hf_embeddings.embed_documents(
    document_chunks
)

hf_time = time.perf_counter() - start_time

In [27]:
print(
    f"{'Vector Dimensions':<25} "
    f"{len(openai_vectors[0]):<20} "
    f"{len(hf_vectors[0]):<20}"
)


Vector Dimensions         1536                 384                 


In [28]:
print(
    f"{'Embedding Time':<25} "
    f"{openai_time:.4f}s{'':<14} "
    f"{hf_time:.4f}s"
)

Embedding Time            1.1771s               0.0271s


## Task 3: OpenAI vs Hugging Face

1. When should you prefer OpenAI embeddings?

- OpenAI embeddings are useful when a managed API is preferred and local model management is undesirable. They provide a simple integration path but introduce API dependency and usage costs.

2. When should you prefer Hugging Face embeddings?

- Hugging Face embeddings are useful when local inference, cost control, privacy, or model customization is important. They require local computational resources and model setup.

3. Cost vs Performance trade-offs

OpenAI

- Easier managed service
- API cost
- External API dependency
- 1536-dimensional vectors

Hugging Face
- Local inference
- No per-request API cost
- Requires local resources
- 384-dimensional vectors

# Part 2: Document Similarity Search (OpenAI Embeddings)

## TAsk 4: Similarity search App ( Core logic )

In [30]:
import numpy as np
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

In [31]:
load_dotenv()

True

In [32]:
document_chunks = [
    "Python is a high-level programming language used for web development, automation, and data science.",
    "Machine learning allows computers to learn patterns from data and make predictions.",
    "RAG combines document retrieval with large language models to provide context-aware answers.",
    "Vector databases store numerical embeddings and allow similarity search.",
    "LangChain is a framework for building applications powered by large language models.",
    "FAISS is a library developed for efficient similarity search over dense vectors."
]

In [36]:
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [37]:
document_embeddings = embedding_model.embed_documents(
    document_chunks
)

document_embeddings = np.array(document_embeddings)

In [38]:
print("Number of chunks:", len(document_chunks))
print("Embedding dimension:", document_embeddings.shape[1])

Number of chunks: 6
Embedding dimension: 1536


In [39]:
def search(query, top_k=3):

    query_embedding = embedding_model.embed_query(query)
    query_embedding = np.array(query_embedding)
    similarities = np.dot(
document_embeddings,
        query_embedding
    ) / (
        np.linalg.norm(document_embeddings, axis=1)
        * np.linalg.norm(query_embedding)
    )

    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []

    for index in top_indices:
        results.append({
            "document": document_chunks[index],
            "score": similarities[index]
        })

    return results

In [40]:
result1 = search("What is machine learning", top_k=3)

In [41]:
result1

[{'document': 'Machine learning allows computers to learn patterns from data and make predictions.',
  'score': 0.6186985928361319},
 {'document': 'LangChain is a framework for building applications powered by large language models.',
  'score': 0.25693952585943347},
 {'document': 'RAG combines document retrieval with large language models to provide context-aware answers.',
  'score': 0.22333194174486865}]

In [ ]:
result2 = search("What is rag", top_k=3)

In [ ]:
result2

In [ ]:
result3 = search("What is rag", top_k=3)

In [ ]:
result3

## TASK 5: Similarity Search with Langchain Abstraction

In [42]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from dotenv import load_dotenv

C:\Users\arunk\AppData\Local\Temp\ipykernel_24912\2033579825.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [43]:
load_dotenv()

True

In [45]:
documents = [
    Document(page_content=chunk)
    for chunk in document_chunks
]

In [46]:
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [47]:
vectorstore = FAISS.from_documents(
    documents,
    embedding_model
)

In [48]:
query = "What is machine learning?"

results = vectorstore.similarity_search(
    query,
    k=3
)

In [49]:
for i, document in enumerate(results, start=1):
    print(document.page_content)

Machine learning allows computers to learn patterns from data and make predictions.
LangChain is a framework for building applications powered by large language models.
RAG combines document retrieval with large language models to provide context-aware answers.


## PART 3: Ollama Embeddings

## TASK 6

In [52]:
from langchain_ollama import OllamaEmbeddings
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv


In [53]:
load_dotenv()

True

In [54]:
ollama_embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

ollama_vectors = ollama_embeddings.embed_documents(
    document_chunks
)

In [55]:
len(openai_vectors[0])

1536

In [56]:
len(ollama_vectors[0])

768

In [58]:
openai_vectors[0][:5]

[-0.007259368896484375,
 0.00907135009765625,
 0.0166473388671875,
 0.0034275054931640625,
 0.044189453125]

In [59]:
ollama_vectors[0][:5]

[-0.003825541, 0.08791872, -0.13752878, -0.06283992, 0.068160884]

# Part 4: Vector Stores

## TASK 7: FAISS Vector Store

In [60]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

In [62]:
documents = [
    Document(page_content=chunk)
    for chunk in document_chunks
]

In [63]:
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [64]:
vectorstore = FAISS.from_documents(
    documents,
    embedding_model
)

In [65]:
query = "How does machine learning work?"

results = vectorstore.similarity_search(
    query,
    k=3
)

In [66]:
for i, doc in enumerate(results, start=1):

    print(f"\nResult {i}:")
    print(doc.page_content)


Result 1:
Machine learning allows computers to learn patterns from data and make predictions.

Result 2:
RAG combines document retrieval with large language models to provide context-aware answers.

Result 3:
Vector databases store numerical embeddings and allow similarity search.


In [67]:
vectorstore.save_local(
    "faiss_index"
)

In [68]:
loaded_vectorstore = FAISS.load_local(
    "faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True
)

In [69]:
print("FAISS index loaded successfully.")

FAISS index loaded successfully.


In [70]:
results = loaded_vectorstore.similarity_search(
    "What is RAG?",
    k=3
)

In [71]:
for i, doc in enumerate(results, start=1):

    print(f"\nResult {i}:")
    print(doc.page_content)


Result 1:
RAG combines document retrieval with large language models to provide context-aware answers.

Result 2:
LangChain is a framework for building applications powered by large language models.

Result 3:
FAISS is a library developed for efficient similarity search over dense vectors.


## TASK 8: ChromaDB Vector Store

In [73]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

In [74]:
documents = [
    Document(page_content=chunk)
    for chunk in document_chunks
]

In [75]:
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [76]:

vectorstore = Chroma(
    collection_name="personal_knowledge",
    embedding_function=embedding_model,
    persist_directory="./chroma_db"
)

In [77]:
vectorstore.add_documents(
    documents
)

print("Documents added to ChromaDB.")

Documents added to ChromaDB.


In [78]:
query = "What is machine learning?"

results = vectorstore.similarity_search(
    query,
    k=3
)

In [79]:
for i, doc in enumerate(results, start=1):

    print(f"\nResult {i}:")
    print(doc.page_content)


Result 1:
Machine learning allows computers to learn patterns from data and make predictions.

Result 2:
LangChain is a framework for building applications powered by large language models.

Result 3:
RAG combines document retrieval with large language models to provide context-aware answers.


## TASK 9: FAISS vs ChromaDB comparison

1. In memory vs persistent storage
- FAISS is primarily a similarity-search/indexing library.
- ChromaDB provides a more database-like persistent workflow.

2. Use cases for FAISS
- if you want fast vector similarity search because it use in memory storage

3. User CAses for ChromaDB
- it is useful when you want a more database-oriented experience around embeddings and documents.

# PART 5: Mini Project Integration

## TASK 10

In [80]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_openai import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import OllamaEmbeddings

from langchain_community.vectorstores import FAISS
from langchain_chroma import Chroma

from langchain_core.documents import Document

In [81]:
EMBEDDING_PROVIDER = "openai"
VECTOR_STORE = "faiss"

TOP_K = 3

In [83]:
with open("data/notes.txt", "r", encoding="utf-8") as file:
    text = file.read()


In [84]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = text_splitter.split_text(text)

In [85]:
documents = [
    Document(
        page_content=chunk,
        metadata={
            "source": "documents.txt",
            "chunk_id": i
        }
    )
    for i, chunk in enumerate(chunks)
]


In [86]:
len(documents)

4

In [87]:
def get_embedding_model(provider):

    if provider == "openai":
        return OpenAIEmbeddings(model="text-embedding-3-small")

    elif provider == "huggingface":
        return HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

    elif provider == "ollama":
        return OllamaEmbeddings(model="nomic-embed-text")

    else:
        raise ValueError(
            "Invalid embedding provider. "
            "Choose: openai, huggingface, ollama"
        )

In [88]:
def create_vector_store(
    documents,
    embedding_model,
    provider,
    store_type
):

    if store_type == "faiss":

        vectorstore = FAISS.from_documents(documents,embedding_model)
        path = f"vector_db/faiss_{provider}"

        os.makedirs("vector_db", exist_ok=True)
        vectorstore.save_local(path)
        return vectorstore


    elif store_type == "chroma":

        path = f"vector_db/chroma_{provider}"

        vectorstore = Chroma(collection_name=f"knowledge_{provider}",embedding_function=embedding_model,persist_directory=path)
        vectorstore.add_documents(documents)
        return vectorstore
    else:
        raise ValueError(
            "Invalid vector store. "
            "Choose: faiss or chroma"
        )        


In [ ]:
embedding_model = get_embedding_model(EMBEDDING_PROVIDER)

In [89]:
vectorstore = create_vector_store(
    documents=documents,
    embedding_model=embedding_model,
    provider=EMBEDDING_PROVIDER,
    store_type=VECTOR_STORE
)

In [90]:
def search(query, top_k=3):

    results = vectorstore.similarity_search(
        query,
        k=top_k
    )

    return results

In [91]:
result_1 = search('What is RAG',TOP_K)

In [92]:
result_1

[Document(id='2902de2b-b60a-4af5-97cd-5554e3cca570', metadata={'source': 'documents.txt', 'chunk_id': 1}, page_content='Generative AI uses machine learning models to generate new content such as text, images, audio, and code.\n\nRetrieval Augmented Generation, commonly called RAG, combines document retrieval with a language model. Relevant documents are retrieved before generating an answer.'),
 Document(id='37197968-0b28-49ca-a30d-38b23bdd478c', metadata={'source': 'documents.txt', 'chunk_id': 3}, page_content='ChromaDB is a vector database commonly used for storing embeddings and building retrieval-based AI applications.\n\nLangChain provides abstractions for building applications using language models, embeddings, retrievers, and vector stores.'),
 Document(id='4d530417-6f95-4c5d-8de0-5c3a81c7774c', metadata={'source': 'documents.txt', 'chunk_id': 2}, page_content='Vector databases store numerical representations of data called embeddings. They allow applications to perform similari

In [93]:
result_2 = search('What is machine learning',TOP_K)

In [96]:
result_2

[Document(id='a12e3a5c-7278-42d1-a8a1-be7d40f01126', metadata={'source': 'documents.txt', 'chunk_id': 0}, page_content='Python is a high-level programming language. It is widely used for web development, automation, data science, and artificial intelligence.\n\nMachine learning is a branch of artificial intelligence that enables computers to learn patterns from data and make predictions.'),
 Document(id='2902de2b-b60a-4af5-97cd-5554e3cca570', metadata={'source': 'documents.txt', 'chunk_id': 1}, page_content='Generative AI uses machine learning models to generate new content such as text, images, audio, and code.\n\nRetrieval Augmented Generation, commonly called RAG, combines document retrieval with a language model. Relevant documents are retrieved before generating an answer.'),
 Document(id='37197968-0b28-49ca-a30d-38b23bdd478c', metadata={'source': 'documents.txt', 'chunk_id': 3}, page_content='ChromaDB is a vector database commonly used for storing embeddings and building retrieva

In [ ]:
result_3 = search('What is vector databases',TOP_K)

In [ ]:
result_3

## Task 11 — Observations & Insights

1. Importance of embeddings in GenAI

- Embeddings convert text into numerical vectors that represent semantic information.
- This allows a retrieval system to search by semantic meaning rather than exact keyword matching.

2. Why vector databases are required
- Vector databases store and index embedding vectors and provide efficient similarity search. 
- Instead of relying only on exact keyword matching, they allow applications to find documents based on semantic similarity. 
- They are especially useful when searching large collections of documents.

3. How this pipeline enables RAG systems
- The pipeline creates the retrieval component of a RAG system. 
- Documents are split into chunks, converted into embeddings, stored in a vector store, and retrieved using similarity search. 
- The retrieved chunks can then be passed as context to an LLM to generate a grounded response.